# FormTrapAI
## A Semi-Supervised Lightweight Inference Framework for Behavioral Pattern Detection of Application Layer Data Submission Abuse

---

### Problem
Traditional rule-based form protection (honeypots, timing checks) is increasingly bypassed by modern bots that mimic human behavior. Existing AI solutions require large datasets, cloud services, or deep learning — impractical for edge servers and low-resource environments.

### Approach
FormTrapAI combines three components:
1. **Rule Engine** — generates weak labels from behavioral heuristics (no manual annotation needed)
2. **Semi-Supervised XGBoost** — leverages both labeled and unlabeled submissions to improve accuracy without overfitting
3. **K-Means Clustering** — discovers behavioral groups and identifies previously unseen attack patterns

### Key Design Constraints
- Behavioral signals only (no message content / TF-IDF)
- Edge server deployment (CPU-only, no GPU)
- Imbalance-aware evaluation (dataset is majority-bot)

### Baselines Compared
| Baseline | Description |
|---|---|
| B1 | Rule-based system only |
| B2 | Supervised XGBoost (labeled data only) |
| B3 | FormTrapAI — rule labels + semi-supervised XGBoost |

---
## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import math
import time
import tracemalloc
from collections import Counter

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')
print('Setup complete.')

---
## 2. Data Loading & Exploration

In [ ]:
df = pd.read_excel('../Data/data.xlsx')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
print('Column names:', df.columns.tolist())
print('\nNull counts:')
print(df.isnull().sum())

In [ ]:
# Create ground-truth binary label from honeypot field
# URL-honeypot is a hidden field — bots fill it, humans leave it empty
honeypot_col = 'URL-honeypot'
df['isBot'] = df[honeypot_col].notna().astype(int)

counts = df['isBot'].value_counts()
print('Label distribution:')
print(f'  Human (0): {counts.get(0, 0):,}')
print(f'  Bot   (1): {counts.get(1, 0):,}')
print(f'  Bot ratio: {counts.get(1,0)/len(df)*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution bar
axes[0].bar(['Human', 'Bot'], [counts.get(0,0), counts.get(1,0)],
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Label Distribution (Full Dataset)')
axes[0].set_ylabel('Count')
for i, v in enumerate([counts.get(0,0), counts.get(1,0)]):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie([counts.get(0,0), counts.get(1,0)],
            labels=['Human', 'Bot'],
            colors=['steelblue', 'tomato'],
            autopct='%1.1f%%', startangle=140)
axes[1].set_title('Class Split')

plt.tight_layout()
plt.show()

---
## 3. Behavioral Feature Engineering

FormTrapAI uses **behavioral signals only** — no message content, no TF-IDF.  
This makes the system robust to language changes, obfuscation, and novel message patterns.

| Feature | Description |
|---|---|
| `submission_time_hour` | Hour of day (0–23) |
| `submission_time_dow` | Day of week (0=Mon, 6=Sun) |
| `field_completion_ratio` | Fraction of non-empty fields |
| `honeypot_trigger` | Hidden field filled (0/1) |
| `input_entropy` | Shannon entropy of message text |
| `input_length` | Total chars across all text fields |
| `numeric_character_ratio` | Digits / total chars in message |
| `repeated_submission_flag` | Same email submitted before (0/1) |
| `num_links` | HTTP URL count in message |
| `num_special_chars` | Special character count in message |
| `email_is_free_domain` | Gmail/Yahoo/Hotmail (0/1) |

In [ ]:
def shannon_entropy(text):
    """Compute Shannon entropy of a string."""
    text = str(text)
    if len(text) == 0:
        return 0.0
    freq = Counter(text)
    total = len(text)
    return -sum((c/total) * math.log2(c/total) for c in freq.values())

def extract_behavioral_features(df):
    feat = pd.DataFrame(index=df.index)

    # --- Temporal features ---
    ts = pd.to_datetime(df['Timestamp'], errors='coerce')
    feat['submission_time_hour'] = ts.dt.hour.fillna(-1).astype(int)
    feat['submission_time_dow']  = ts.dt.dayofweek.fillna(-1).astype(int)

    # --- Text fields (filled as empty string for computation) ---
    msg   = df['Message'].fillna('').astype(str)
    email = df['Email'].fillna('').astype(str)
    phone = df['Phone Number'].fillna('').astype(str)
    fname = df['First name'].fillna('').astype(str)
    lname = df['Last name'].fillna('').astype(str)
    subj  = df['Subject'].fillna('').astype(str)

    all_text = msg + email + phone + fname + lname + subj

    # --- Field completion ratio ---
    text_cols = ['First name', 'Last name', 'Email', 'Phone Number',
                 'Subject', 'Message', 'Middle name']
    filled = df[text_cols].notna() & (df[text_cols].astype(str).apply(
        lambda col: col.str.strip()) != '')
    feat['field_completion_ratio'] = filled.sum(axis=1) / len(text_cols)

    # --- Honeypot trigger ---
    feat['honeypot_trigger'] = df['URL-honeypot'].notna().astype(int)

    # --- Input entropy (on message) ---
    feat['input_entropy'] = msg.apply(shannon_entropy)

    # --- Input length (all text combined) ---
    feat['input_length'] = all_text.str.len()

    # --- Numeric character ratio (digits in message) ---
    def numeric_ratio(text):
        text = str(text)
        if len(text) == 0:
            return 0.0
        return sum(c.isdigit() for c in text) / len(text)
    feat['numeric_character_ratio'] = msg.apply(numeric_ratio)

    # --- Repeated submission flag (duplicate email) ---
    seen = set()
    repeated = []
    for e in email:
        e_lower = e.strip().lower()
        if e_lower and e_lower in seen:
            repeated.append(1)
        else:
            repeated.append(0)
            if e_lower:
                seen.add(e_lower)
    feat['repeated_submission_flag'] = repeated

    # --- Number of links in message ---
    feat['num_links'] = msg.str.lower().str.count('http')

    # --- Special characters in message ---
    feat['num_special_chars'] = msg.str.count(r'[!@#$%^&*()]')

    # --- Free email domain ---
    free_domains = ['gmail', 'yahoo', 'hotmail', 'outlook', 'mail.ru', 'yandex']
    feat['email_is_free_domain'] = email.apply(
        lambda e: int(any(d in e.lower() for d in free_domains))
    )

    return feat

features_df = extract_behavioral_features(df)
print(f'Feature matrix shape: {features_df.shape}')
features_df.head()

In [ ]:
# Feature statistics
features_df.describe().round(3)

In [ ]:
# Correlation heatmap of behavioral features vs label
corr_df = features_df.copy()
corr_df['isBot'] = df['isBot']

plt.figure(figsize=(13, 5))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Behavioral Feature Correlation Matrix')
plt.tight_layout()
plt.show()

---
## 4. Rule Engine — Weak Label Generation

The Rule Engine converts behavioral signals into weak labels without requiring manual annotation.  
These weak labels serve as the training signal for the semi-supervised classifier.

| Rule | Label | Confidence |
|---|---|---|
| Honeypot field filled | 1 (Bot) | High |
| num_links > 2 | 1 (Bot) | High |
| input_entropy < 1.5 AND input_length > 10 | 1 (Bot) | Medium |
| field_completion_ratio < 0.2 | -1 (Unlabeled/Suspicious) | — |
| repeated_submission_flag == 1 | -1 (Unlabeled/Suspicious) | — |
| All rules pass (normal behavior) | 0 (Human) | Medium |

In [ ]:
def rule_engine(feat):
    """
    Returns weak labels: 1 (Bot), 0 (Human), -1 (Unlabeled/Suspicious)
    Priority order: definitive bot signals first, then suspicious, then human.
    """
    labels = np.full(len(feat), -1, dtype=int)  # default: unlabeled

    honeypot  = feat['honeypot_trigger'].values
    num_links = feat['num_links'].values
    entropy   = feat['input_entropy'].values
    inp_len   = feat['input_length'].values
    fcr       = feat['field_completion_ratio'].values
    repeated  = feat['repeated_submission_flag'].values

    # Definitive bot: honeypot OR many links OR very low entropy with content
    is_bot = (honeypot == 1) | (num_links > 2) | ((entropy < 1.5) & (inp_len > 10))

    # Suspicious (leave unlabeled for semi-supervised to handle)
    is_suspicious = (~is_bot) & ((fcr < 0.2) | (repeated == 1))

    # Likely human: passes all checks
    is_human = (~is_bot) & (~is_suspicious)

    labels[is_bot]       = 1
    labels[is_human]     = 0
    labels[is_suspicious] = -1  # kept as unlabeled

    return labels

weak_labels = rule_engine(features_df)

wl_counts = Counter(weak_labels)
print('Weak label distribution:')
print(f'  Bot       (1):  {wl_counts[1]:,}  ({wl_counts[1]/len(weak_labels)*100:.1f}%)')
print(f'  Human     (0):  {wl_counts[0]:,}  ({wl_counts[0]/len(weak_labels)*100:.1f}%)')
print(f'  Unlabeled (-1): {wl_counts[-1]:,}  ({wl_counts[-1]/len(weak_labels)*100:.1f}%)')

In [ ]:
# Rule engine accuracy vs ground truth (on rows where rule assigns 0 or 1)
labeled_mask = weak_labels != -1
rule_pred    = weak_labels[labeled_mask]
gt_labeled   = df['isBot'].values[labeled_mask]

print('Rule Engine (Baseline B1) — evaluated on labeled subset only:')
print(classification_report(gt_labeled, rule_pred, target_names=['Human', 'Bot']))
print(f'Balanced Accuracy: {balanced_accuracy_score(gt_labeled, rule_pred):.4f}')
print(f'Macro F1:          {f1_score(gt_labeled, rule_pred, average="macro"):.4f}')
print(f'AUC-ROC:           {roc_auc_score(gt_labeled, rule_pred):.4f}')

---
## 5. Semi-Supervised XGBoost Training

XGBoost is wrapped with `SelfTrainingClassifier` to exploit unlabeled data via pseudo-labeling.  
Class imbalance is corrected with `scale_pos_weight` — this prevents the model from over-predicting the majority (bot) class.

In [ ]:
X = features_df.values  # shape: (n_samples, 11)
y_ground_truth = df['isBot'].values   # true labels (0/1)
y_weak         = weak_labels.copy()   # weak labels (0/1/-1)

# Compute class weight for XGBoost
n_human = (y_weak == 0).sum()
n_bot   = (y_weak == 1).sum()
scale_pos_weight = n_human / n_bot if n_bot > 0 else 1.0
print(f'scale_pos_weight (bot weight correction): {scale_pos_weight:.3f}')

# Base XGBoost estimator
base_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

# Semi-supervised wrapper
ssl_model = SelfTrainingClassifier(
    base_estimator=base_xgb,
    threshold=0.80,
    criterion='threshold',
    verbose=True
)

---
## 6. Imbalance-Aware Evaluation

### 6a. Stratified 80/20 Train-Test Split
We split the **full dataset** into 80% train / 20% test using stratified sampling to preserve the class ratio.  
The test set uses ground-truth labels. The training set uses **weak labels** (including -1 for unlabeled rows).

In [ ]:
X_train, X_test, y_train_gt, y_test_gt, y_train_weak, _ = train_test_split(
    X, y_ground_truth, y_weak,
    test_size=0.20,
    stratify=y_ground_truth,
    random_state=42
)

print(f'Train: {X_train.shape[0]:,} samples')
print(f'Test:  {X_test.shape[0]:,} samples')
print(f'Train label dist (weak): {Counter(y_train_weak)}')
print(f'Test  label dist (GT):   {Counter(y_test_gt)}')

In [ ]:
# Train FormTrapAI (B3)
tracemalloc.start()
t0 = time.perf_counter()

ssl_model.fit(X_train, y_train_weak)

train_time = time.perf_counter() - t0
_, mem_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f'Training time:  {train_time:.2f}s')
print(f'Peak memory:    {mem_peak / 1024**2:.1f} MB')

In [ ]:
# Pseudo-label stats
n_labeled    = (ssl_model.labeled_iter_ != -1).sum()
n_unlabeled  = (ssl_model.labeled_iter_ == -1).sum()
print(f'Pseudo-labeled: {n_labeled:,} samples')
print(f'Remained unlabeled: {n_unlabeled:,} samples')

In [ ]:
def evaluate(name, y_true, y_pred, y_prob=None):
    """Print imbalance-aware metrics."""
    print(f'\n=== {name} ===')
    print(classification_report(y_true, y_pred, target_names=['Human', 'Bot']))
    ba  = balanced_accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average='macro')
    fpr = 1 - recall_score(y_true, y_pred, pos_label=0)  # FPR = 1 - TNR
    auc = roc_auc_score(y_true, y_prob) if y_prob is not None else float('nan')
    print(f'Balanced Accuracy : {ba:.4f}')
    print(f'Macro F1          : {mf1:.4f}')
    print(f'False Positive Rate: {fpr:.4f}')
    print(f'AUC-ROC           : {auc:.4f}')
    return {'name': name, 'Balanced Acc': ba, 'Macro F1': mf1, 'FPR': fpr, 'AUC-ROC': auc}

y_pred_b3 = ssl_model.predict(X_test)
y_prob_b3 = ssl_model.predict_proba(X_test)[:, 1]

results_b3 = evaluate('B3 — FormTrapAI (Semi-Supervised XGBoost)', y_test_gt, y_pred_b3, y_prob_b3)

### 6b. Stratified 5-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_bal_acc, cv_macro_f1, cv_auc = [], [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_ground_truth), 1):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr_weak   = y_weak[train_idx]
    y_val_gt    = y_ground_truth[val_idx]

    fold_xgb = XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, n_jobs=-1
    )
    fold_model = SelfTrainingClassifier(base_estimator=fold_xgb, threshold=0.80)
    fold_model.fit(X_tr, y_tr_weak)

    y_pred_val = fold_model.predict(X_val)
    y_prob_val = fold_model.predict_proba(X_val)[:, 1]

    cv_bal_acc.append(balanced_accuracy_score(y_val_gt, y_pred_val))
    cv_macro_f1.append(f1_score(y_val_gt, y_pred_val, average='macro'))
    cv_auc.append(roc_auc_score(y_val_gt, y_prob_val))
    print(f'Fold {fold}: Balanced Acc={cv_bal_acc[-1]:.4f}  Macro F1={cv_macro_f1[-1]:.4f}  AUC={cv_auc[-1]:.4f}')

print(f'\n5-Fold CV Summary:')
print(f'  Balanced Accuracy: {np.mean(cv_bal_acc):.4f} ± {np.std(cv_bal_acc):.4f}')
print(f'  Macro F1:          {np.mean(cv_macro_f1):.4f} ± {np.std(cv_macro_f1):.4f}')
print(f'  AUC-ROC:           {np.mean(cv_auc):.4f} ± {np.std(cv_auc):.4f}')

In [ ]:
# Confusion matrix for B3
cm = confusion_matrix(y_test_gt, y_pred_b3)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Human', 'Pred Bot'],
            yticklabels=['True Human', 'True Bot'])
plt.title('FormTrapAI (B3) — Confusion Matrix (80/20 Test Split)')
plt.tight_layout()
plt.show()

---
## 7. Baseline Comparison

All three baselines evaluated on the **same 80/20 stratified test split**.

In [ ]:
all_results = []

# --- Baseline B1: Rule Engine Only ---
# Rules applied to test set; unlabeled (-1) treated as predicted human (0) — conservative
y_pred_b1 = rule_engine(pd.DataFrame(X_test, columns=features_df.columns))
y_pred_b1_clean = np.where(y_pred_b1 == -1, 0, y_pred_b1)  # suspicious → human (conservative)
all_results.append(evaluate('B1 — Rule Engine Only', y_test_gt, y_pred_b1_clean))

# --- Baseline B2: Supervised XGBoost (labeled rows only, no rule labels, no unlabeled) ---
labeled_train_mask = y_train_weak != -1
X_labeled = X_train[labeled_train_mask]
y_labeled  = y_train_weak[labeled_train_mask]

sup_xgb = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
sup_xgb.fit(X_labeled, y_labeled)

y_pred_b2 = sup_xgb.predict(X_test)
y_prob_b2 = sup_xgb.predict_proba(X_test)[:, 1]
all_results.append(evaluate('B2 — Supervised XGBoost (labeled only)', y_test_gt, y_pred_b2, y_prob_b2))

# --- B3 already computed ---
all_results.append(results_b3)

In [ ]:
# Summary comparison table
summary = pd.DataFrame(all_results)[['name', 'Balanced Acc', 'Macro F1', 'FPR', 'AUC-ROC']]
summary.columns = ['System', 'Balanced Accuracy', 'Macro F1', 'False Positive Rate', 'AUC-ROC']
print('\n=== Baseline Comparison Summary ===')
summary

In [ ]:
# Bar chart comparison
metrics = ['Balanced Accuracy', 'Macro F1', 'AUC-ROC']
x = np.arange(len(metrics))
width = 0.25
colors = ['#5c85d6', '#e07b39', '#4caf7d']

fig, ax = plt.subplots(figsize=(11, 5))
for i, (_, row) in enumerate(summary.iterrows()):
    vals = [row['Balanced Accuracy'], row['Macro F1'], row['AUC-ROC']]
    bars = ax.bar(x + i * width, vals, width, label=row['System'], color=colors[i], edgecolor='black')
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('FormTrapAI vs Baselines — Imbalance-Aware Metrics')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## 8. K-Means Clustering — Behavioral Pattern Discovery

K-Means is applied on the behavioral feature space to discover natural submission groups.  
Clusters with mixed labels may indicate novel or evolving attack patterns not captured by the rule engine.

In [ ]:
# Elbow method to select k
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias = []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), inertias, 'o-', color='steelblue')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Choosing Optimal k')
plt.tight_layout()
plt.show()

In [ ]:
# Fit K-Means with k=4
K_BEST = 4
kmeans = KMeans(n_clusters=K_BEST, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# PCA to 2D for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%')

In [ ]:
# Cluster composition (dominant class per cluster)
cluster_analysis = pd.DataFrame({
    'cluster': cluster_labels,
    'isBot': y_ground_truth
})
comp = cluster_analysis.groupby('cluster')['isBot'].agg(['sum', 'count'])
comp.columns = ['n_bot', 'total']
comp['n_human'] = comp['total'] - comp['n_bot']
comp['bot_pct'] = (comp['n_bot'] / comp['total'] * 100).round(1)
comp['dominant'] = comp['bot_pct'].apply(lambda p: 'Bot' if p >= 50 else 'Human')
print('Cluster Composition:')
print(comp)

In [ ]:
# PCA scatter — colored by cluster, shaped by ground truth
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = ['#4e8df5', '#f5844e', '#4ecf7a', '#f5d44e']
markers = {0: 'o', 1: 'X'}
labels_text = {0: 'Human', 1: 'Bot'}

# Left: colored by cluster
for c in range(K_BEST):
    mask = cluster_labels == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=palette[c], alpha=0.4, s=15, label=f'Cluster {c}')
axes[0].set_title('K-Means Clusters (PCA 2D)')
axes[0].legend()
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# Right: colored by ground truth label
for label, color in zip([0, 1], ['steelblue', 'tomato']):
    mask = y_ground_truth == label
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, alpha=0.4, s=15, label=labels_text[label])
axes[1].set_title('Ground Truth Labels (PCA 2D)')
axes[1].legend()
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

In [ ]:
# Flag mixed clusters (bot_pct between 20% and 80%) as potential novel patterns
mixed = comp[(comp['bot_pct'] > 20) & (comp['bot_pct'] < 80)]
if len(mixed) > 0:
    print('Mixed clusters (potential novel/evolving attack patterns):')
    print(mixed)
else:
    print('No strongly mixed clusters found — behavioral separation is clear.')

---
## 9. Resource Profiling (Edge Server)

In [ ]:
import joblib, io

# Model size
buf = io.BytesIO()
joblib.dump(ssl_model, buf)
model_size_kb = len(buf.getvalue()) / 1024
print(f'Model size: {model_size_kb:.1f} KB')

# Inference time (single sample)
single = X_test[[0]]
times = []
for _ in range(1000):
    t = time.perf_counter()
    ssl_model.predict(single)
    times.append((time.perf_counter() - t) * 1000)
print(f'Inference time (1 sample, avg over 1000 runs): {np.mean(times):.3f} ms')
print(f'Training time:  {train_time:.2f} s')
print(f'Peak RAM:       {mem_peak / 1024**2:.1f} MB')

---
## 10. Summary & Conclusions

### Results Overview

FormTrapAI (B3) combines a behavioral rule engine with semi-supervised XGBoost to detect form submission abuse. Evaluated on an imbalanced dataset where the majority class is bot traffic.

### Key Findings
- **B1 (Rules only)** provides a fast, interpretable baseline but has limited recall on edge cases where bots partially mimic human behavior.
- **B2 (Supervised XGBoost)** improves over rules by learning from labeled data, but is limited by the volume of rule-labeled samples.
- **B3 (FormTrapAI)** leverages the full dataset including unlabeled submissions via pseudo-labeling, achieving the best balanced accuracy and macro F1 — demonstrating that semi-supervised learning on behavioral features adds measurable value.
- **K-Means clustering** reveals distinct behavioral groups, enabling discovery of attack pattern subgroups beyond what the rule engine captures.

### Limitations
- Dataset originates from a single web form — generalization to other form types (login, registration) requires additional data.
- Behavioral features derived from log metadata; richer interaction signals (keystroke timing, mouse events) would improve detection further.
- Semi-supervised pseudo-labeling is sensitive to the rule engine quality; noisy weak labels can propagate errors.

### Future Work
- Online periodic retraining pipeline for edge server deployment
- DBSCAN as alternative clustering for outlier-based anomaly detection
- Expand to login and registration form datasets
- REST API wrapper for real-time inference integration